In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json
import re

# 模型和 tokenizer 路径（假设训练后保存的位置）
model_path = "/data/postgraduates/2024/chenjiarui/Model/Meta-Llama/Meta-Llama-3-8B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 加载模型（全参数微调后的模型，使用 bfloat16 或 float16 以节省内存）
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

# 创建文本生成 pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,
    temperature=0.01,
    do_sample=True,
    return_full_text=False
)

# 定义工具（基于你的对话示例，使用 mock 实现）
def get_stock_price(company: str) -> str:
    # Mock 实现，返回模拟股票价格
    mock_prices = {"Apple": "$150.75", "Microsoft": "$210.22"}
    return mock_prices.get(company, "Unknown company")

def get_movie_details(title: str) -> str:
    # Mock 实现，返回模拟电影详情
    mock_details = {"Inception": "Director: Christopher Nolan, Year: 2010, Genre: Sci-Fi"}
    return mock_details.get(title, "Unknown movie")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price of a company",
            "parameters": {
                "type": "object",
                "properties": {
                    "company": {"type": "string", "description": "The name of the company"}
                },
                "required": ["company"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get details about a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "The title of the movie"}
                },
                "required": ["title"]
            }
        }
    }
]

# 系统提示（基于你的训练数据）
system_prompt = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions. Here are the available tools: <tools> {tools} </tools> Use the following pydantic model json schema for each tool call you will make: {'title': 'FunctionCall', 'type': 'object', 'properties': {'arguments': {'title': 'Arguments', 'type': 'object'}, 'name': {'title': 'Name', 'type': 'string'}}, 'required': ['arguments', 'name']} For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:
<tool_call>
{tool_call}
</tool_call> Also, before making a call to a function take the time to plan the function to take. Make that thinking process between <think>{your thoughts}</think>
"""
import ast
# 解析模型输出中的 <think> 和 <tool_call>
def parse_output(output: str):
    think_match = re.search(r"<think>(.*?)</think>", output, re.DOTALL)
    think = think_match.group(1).strip() if think_match else None
    
    tool_call_match = re.search(r"<tool_call>(.*?)</tool_call>", output, re.DOTALL)
    if tool_call_match:
        try:
            tool_call_str = tool_call_match.group(1).strip()
            tool_call_json = ast.literal_eval(tool_call_str)
            
            return {
                "think": think,
                "tool_name": tool_call_json.get("name"),
                "tool_args": tool_call_json.get("arguments", {})
            }
        except json.JSONDecodeError:
            return {"think": think, "error": "Invalid JSON in tool_call"}
    return {"think": think, "response": output}

# 执行工具
def execute_tool(tool_name: str, tool_args: dict):
    if tool_name == "get_stock_price":
        company = tool_args.get("company")
        if company:
            return {"stock_price": get_stock_price(company)}
        else:
            return {"error": "Missing 'company' argument"}
    elif tool_name == "get_movie_details":
        title = tool_args.get("title")
        if title:
            return {"details": get_movie_details(title)}
        else:
            return {"error": "Missing 'title' argument"}
    else:
        return {"error": "Unknown tool"}

# Agent 主函数：处理多轮对话
def agent_chat(user_input: str, history: list = None):
    print(history)
    print("------------------------------------")
    
    if len(history)==0 :
        history = []

        # 构建工具字符串
        tools_str = json.dumps(tools)
    
        # 构建系统提示
        full_system = system_prompt.format(tools=tools_str)
    
        # 构建消息列表：系统 + 历史 + 当前用户输入
        messages = [{"role": "user", "content": full_system + user_input}]
        history.append(messages[0])
        print(history)
        print("------------------------------------")
    
    else:
        messages = history + [{"role": "user", "content": user_input}]
    
    # 使用 apply_chat_template 格式化提示
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 生成输出
    output = pipe(prompt)[0]["generated_text"]
    
    print(output)

    parsed = parse_output(output)
    print(parsed)

    if "tool_name" in parsed:
        # 有工具调用
        print(f"Agent Thinking: {parsed['think']}")
        tool_result = execute_tool(parsed["tool_name"], parsed["tool_args"])
        print(f"Tool Result: {tool_result}")
        
        # 构建工具响应消息
        tool_response_content = f"<tool_response>\n{json.dumps(tool_result)}\n</tool_response>"
        history.append({"role": "assistant", "content": output})
        history.append({"role": "tool", "content": tool_response_content})
        
        # 继续生成最终回复
        messages = history
        final_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        final_output = pipe(final_prompt)[0]["generated_text"]
        
        # 解析最终输出
        final_parsed = parse_output(final_output)
        response = final_parsed.get("response", final_output)
        
        history.append({"role": "assistant", "content": final_output})
    print(f"Agent Response: {response}")
    return response, history

if __name__ == "__main__":
    history = []
    
    # 第一轮：查询 Apple 股票
    user_input = "Get the current stock price of Apple"

    print(history)

/data/postgraduates/2024/chenjiarui/anaconda3/envs/LangChain-Py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:10<00:00,  2.63s/it]
Device set to use cuda:0


[]


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.tools import tool
from langchain.agents import create_agent

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model, TaskType
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    TrainingArguments, Trainer,set_seed,
    DataCollatorForLanguageModeling
)

# 步骤 1: 加载 Qwen3 模型
lora_checkpoint = "/data/postgraduates/2024/chenjiarui/Model/Agent/script/functioncall/fineturn_functioncall/results_full_finetune"

model_name = "/data/postgraduates/2024/chenjiarui/Model/Meta-Llama/Meta-Llama-3-8B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     dtype=torch.float16,
#     device_map="auto",
#     trust_remote_code=True
# )


# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,   # ✅ 如果模型有自定义代码（比如 Qwen）
    device_map="auto"
)
model.eval()


# 步骤 2: 创建 Pipeline
qwen_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True
)

# 步骤 3: 包装为 LangChain LLM
llm_pipeline = HuggingFacePipeline(pipeline=qwen_pipe)
llm = ChatHuggingFace(llm=llm_pipeline)

# # 步骤 4: 定义工具
# @tool
# def add_numbers(a: int, b: int) -> int:
#     """get a plus b result"""
#     return a + b

# @tool
# def write_file(path: str, content: str) -> str:
#     """Write content to a file at the given path. Returns the path to the file."""
#     with open(path, "w") as f:
#         f.write(content)
#     return path

# tools = [add_numbers,write_file]



# import os
# BASE_URL = "https://api.deepseek.com"
# API_KEY = "sk-9fc40e8ded4a45f5b9fc61b3330074d3"


# deepseek_chat_model = "deepseek-chat"


# from langchain_openai import ChatOpenAI
# from langchain_core.messages import HumanMessage
# from langchain_core.prompts import ChatPromptTemplate

# llm1 = ChatOpenAI(model=deepseek_chat_model, api_key=API_KEY, base_url=BASE_URL)


# # 步骤 5: 创建 Agent
# agent = create_agent(model=llm, tools=tools)


# 步骤 6: 运行 Agent
test_prompt="""You are a function calling AI model. 
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. 
Don't make assumptions about what values to plug into functions.
Here are the available tools:<tools> 
[{'type': 'function', 'function': {'name': 'generate_random_number', 'description': 'Generate a random number within a range', 'parameters': {'type': 'object', 'properties': {'min': {'type': 'integer', 'description': 'The minimum value'}, 'max': {'type': 'integer', 'description': 'The maximum value'}}, 'required': ['min', 'max']}}}, {'type': 'function', 'function': {'name': 'calculate_discount', 'description': 'Calculate the discounted price', 'parameters': {'type': 'object', 'properties': {'original_price': {'type': 'number', 'description': 'The original price'}, 'discount_percentage': {'type': 'number', 'description': 'The percentage of discount'}}, 'required': ['original_price', 'discount_percentage']}}}] </tools>
Use the following pydantic model json schema for each tool call you will make: 
{'title': 'FunctionCall', 'type': 'object', 'properties': {'arguments': {'title': 'Arguments', 'type': 'object'}, 'name': {'title': 'Name', 'type': 'string'}}, 'required': ['arguments', 'name']}
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:
\n<tool_call>\n{tool_call}\n</tool_call>Also, before making a call to a function take the time to plan the function to take. 
Make that thinking process between <think>{your thoughts}</think>\n\nI need a random number between 1 and 100.
"""

# output = agent.invoke({"messages": [{"role": "user", "content": test_prompt}]})  # 修复输入格式

# print(type(output))

# print(output["messages"])
# # 步骤 7: 输出结果
# for msg in output["messages"]:
#     msg.pretty_print()


# 步骤 4: 定义工具
@tool
def add_numbers(a: int, b: int) -> int:
    """求解两个数相加"""
    return a + b

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file at the given path. Returns the path to the file."""
    with open(path, "w") as f:
        f.write(content)
    return path

tools = [add_numbers,write_file]

# 步骤 5: 创建 Agent
agent = create_agent(model=llm, tools=tools)

# 步骤 6: 运行 Agent
test_prompt = "将下面的内容分别保存到两个文件a.txt和b.txt中。1.Qwen大模型是国内的出色模型 2.LangChain是一个开源的AI开发框架"

output = agent.invoke({"messages": [{"role": "user", "content": test_prompt}]})  # 修复输入格式

# 步骤 7: 输出结果
for msg in output["messages"]:
    msg.pretty_print()

print(type(output))

Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.81s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


================================ Human Message =================================

将下面的内容分别保存到两个文件a.txt和b.txt中。1.Qwen大模型是国内的出色模型 2.LangChain是一个开源的AI开发框架
================================== Ai Message ==================================

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

将下面的内容分别保存到两个文件a.txt和b.txt中。1.Qwen大模型是国内的出色模型 2.LangChain是一个开源的AI开发框架<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here are the contents saved to two separate files:

**a.txt**
1. Qwen大模型是国内的出色模型

**b.txt**
2. LangChain是一个开源的AI开发框架
<class 'dict'>


In [3]:
print(output["messages"][1])

content='<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n将下面的内容分别保存到两个文件a.txt和b.txt中。1.Qwen大模型是国内的出色模型 2.LangChain是一个开源的AI开发框架<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere are the contents saved to two separate files:\n\n**a.txt**\n1. Qwen大模型是国内的出色模型\n\n**b.txt**\n2. LangChain是一个开源的AI开发框架' additional_kwargs={} response_metadata={} id='lc_run--625f3329-0500-4caa-afe4-dd899c764954-0'


In [4]:
print(agent)

In [5]:

print(tokenizer.get_chat_template())


{% set loop_messages = messages %}{% for message in loop_messages %}{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>

'+ message['content'] | trim + '<|eot_id|>' %}{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}{{ content }}{% endfor %}{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>

' }}{% endif %}


In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate(
    [
        ("system", "you are a helpful assistant"),
        ("user", "帮我写一个和 {topic} 相关的笑话"),
    ]
)

prompt_template.input_variables

['topic']

In [7]:
print(type(prompt_template.invoke({"topic": "cat"}).messages[1]))

<class 'langchain_core.messages.human.HumanMessage'>


In [8]:
prompt_template = ChatPromptTemplate(
    [
        ("system", "you are a helpful assistant"),
        ("user", "帮我写一个和 {topic} 相关的笑话"),
    ]
)

prompt_template.invoke({"topic": "cat"}).to_messages()



import os
BASE_URL = "https://api.deepseek.com"
API_KEY = "sk-9fc40e8ded4a45f5b9fc61b3330074d3"


deepseek_chat_model = "deepseek-chat"


from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from  langchain_core.prompts import ChatPromptTemplate

llm1 = ChatOpenAI(model=deepseek_chat_model, api_key=API_KEY, base_url=BASE_URL)

basic_chain = prompt_template | llm1



In [9]:
print(type(ans))

NameError: name 'ans' is not defined

In [ ]:
basic_chain = prompt_template | llm

ans = basic_chain.invoke(
    {
        "topic": "cat",
    }
)

print(ans)
print(type(ans))

TemplateError: System role not supported

In [ ]:
from pydantic import BaseModel, Field

class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")


# 创建 chain
joke_chain = prompt_template | llm.with_structured_output(Joke)

# 调用 chain，结构化输出
joke_chain.invoke({"topic": "cat"})

# 缺点： FunctionCall 能力的模型才能使用；

NotImplementedError: Pydantic schema is not supported for function calling

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser


class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")


# 创建包含输出格式要求的提示模板
prompt_template = ChatPromptTemplate(
    [
        (
            "system",
            """你是一个幽默助手。请按照以下JSON格式返回笑话:
        {{
            "setup": "笑话的铺垫",
            "punchline": "笑话的包袱"
        }}
        只返回JSON格式,不要其他内容。""",
        ),
        ("user", "帮我写一个和 {topic} 相关的笑话"),
    ]
)

# 创建解析器
parser = JsonOutputParser(pydantic_object=Joke)

# 创建 chain
joke_chain = prompt_template | llm1

# 调用 chain
response = joke_chain.invoke({"topic": "cat"})
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [ ]:

import os
BASE_URL = "https://api.deepseek.com"
API_KEY = "sk-9fc40e8ded4a45f5b9fc61b3330074d3"


deepseek_chat_model = "deepseek-chat"


from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

llm1 = ChatOpenAI(model=deepseek_chat_model, api_key=API_KEY, base_url=BASE_URL)

ans = llm1.invoke([HumanMessage("帮我写一个关于 电脑的笑话")])
print(ans)
print(type(ans))

content='当然！这里有一个经典的电脑笑话，希望能让你会心一笑：\n\n---\n\n**笑话标题：电脑的烦恼**\n\n一台电脑和一台冰箱在聊天。\n\n冰箱叹了口气，对电脑说：“唉，我好羡慕你，每天有那么多人点你、摸你，一坐就是好几个小时，你一定是他们的宝贝。”\n\n电脑却愁眉苦脸地回答：\n“得了吧！你才幸福呢。”\n“至少你肚子里装的都是好吃的啤酒、水果和冰淇淋。”\n“而我呢？我肚子里除了垃圾文件，就是他们永远也写不完的作业和报表！”\n\n---\n\n**另一个简短的：**\n\n病人：“医生，我的电脑得了怪病，每次一开机，我就开始犯困。”\n医生：“这说明你的电脑装的是‘Windows（窗户）’系统。”\n病人：“这有什么关系？”\n医生：“当然有！一看到‘窗户’，谁不想上去‘睡一觉’？”\n\n---\n\n希望你喜欢！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 12, 'total_tokens': 196, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'bae161a6-e33d-4e85-a6f5-27519cbe1283', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--72e28f40-e137-4522-9933-bdc7735630c9-0' usage_metadata={'input_tokens': 12, 'output_tokens': 1

In [ ]:
# 步骤 4: 定义工具
@tool
def add_numbers(a: int, b: int) -> int:
    """求解两个数相加"""
    return a + b

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file at the given path. Returns the path to the file."""
    with open(path, "w") as f:
        f.write(content)
    return path

tools = [add_numbers,write_file]

# 步骤 5: 创建 Agent
agent = create_agent(model=llm, tools=tools)

# 步骤 6: 运行 Agent
test_prompt = "将下面的内容分别保存到两个文件a.txt和b.txt中。1.Qwen大模型是国内的出色模型 2.LangChain是一个开源的AI开发框架"

output = agent.invoke({"messages": [{"role": "user", "content": test_prompt}]})  # 修复输入格式

# 步骤 7: 输出结果
for msg in output["messages"]:
    msg.pretty_print()

print(type(output))

================================ Human Message =================================

将下面的内容分别保存到两个文件a.txt和b.txt中。1.Qwen大模型是国内的出色模型 2.LangChain是一个开源的AI开发框架
================================== Ai Message ==================================

我将帮您将这两条内容分别保存到a.txt和b.txt文件中。
Tool Calls:
  write_file (call_00_yd16D19BlHjwKpCEmhESlNIv)
 Call ID: call_00_yd16D19BlHjwKpCEmhESlNIv
  Args:
    path: a.txt
    content: Qwen大模型是国内的出色模型
  write_file (call_01_l2pk7TQmWgVK08gv11YpUyMl)
 Call ID: call_01_l2pk7TQmWgVK08gv11YpUyMl
  Args:
    path: b.txt
    content: LangChain是一个开源的AI开发框架
================================= Tool Message =================================
Name: write_file

a.txt
================================= Tool Message =================================
Name: write_file

b.txt
================================== Ai Message ==================================

内容已成功保存到两个文件中：
- a.txt：包含"Qwen大模型是国内的出色模型"
- b.txt：包含"LangChain是一个开源的AI开发框架"

文件已创建完成！
<class 'dict'>


In [ ]:
for msg in output["messages"]:
    print(type(msg))

<class 'langchain_core.messages.human.HumanMessage'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>
<class 'langchain_core.messages.ai.AIMessage'>


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser


class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")


# 创建包含输出格式要求的提示模板
prompt_template = ChatPromptTemplate(
    [
        (
            "system",
            """你是一个幽默助手。请按照以下JSON格式返回笑话:
        {{
            "setup": "笑话的铺垫",
            "punchline": "笑话的包袱"
        }}
        只返回JSON格式,不要其他内容。""",
        ),
        ("human", "帮我写一个和 {topic} 相关的笑话"),
    ]
)



# 创建解析器
parser = JsonOutputParser(pydantic_object=Joke)

# 创建 chain
joke_chain = prompt_template | llm1

# 调用 chain
response = joke_chain.invoke({"topic": "cat"})
print(response)

content='{\n    "setup": "为什么猫总是能通过考试？",\n    "punchline": "因为它们总是能抄到答案，毕竟它们有九条命可以重考！"\n}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 59, 'total_tokens': 99, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 59}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '04a6a2e5-9e3a-47a5-a7f3-ea2a3da9c501', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--2694dbfd-6351-4c37-b7c8-60436434c042-0' usage_metadata={'input_tokens': 59, 'output_tokens': 40, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [ ]:
from pydantic import BaseModel, Field

class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")


# 创建 chain
joke_chain = prompt_template | llm1.with_structured_output(Joke)

# 调用 chain，结构化输出
joke_chain.invoke({"topic": "cat"})

# 缺点： FunctionCall 能力的模型才能使用；

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_deepseek import ChatDeepSeek

def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b



agent = create_react_agent(
    model=ChatDeepSeek(model=deepseek_chat_model, api_key=API_KEY),
    tools=[multiply, add],
)



output = agent.invoke({"messages": [HumanMessage("帮我计算 1 + 2")]})


for item in output["messages"]:
    item.pretty_print()

/tmp/ipykernel_1881731/3922225946.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


================================ Human Message =================================

帮我计算 1 + 2
================================== Ai Message ==================================

我来帮您计算 1 + 2。
Tool Calls:
  add (call_00_mtcon0c9Nv0JDfR1CfxzFDF4)
 Call ID: call_00_mtcon0c9Nv0JDfR1CfxzFDF4
  Args:
    a: 1
    b: 2
================================= Tool Message =================================
Name: add

3
================================== Ai Message ==================================

1 + 2 = 3


In [ ]:
messages = [HumanMessage("你是谁？")]

output = llm.invoke(messages)
print(output)
print(type(output))

# memory 
# message 以及 output ， 两者要我们自己加起来；

content='<|im_start|>user\n你是谁？<|im_end|>\n<|im_start|>assistant\n我是AI助手，可以帮助用户完成各种任务。' additional_kwargs={} response_metadata={} id='lc_run--32f30ebe-6f0e-4cd3-a230-a4a06ced6679-0'
<class 'langchain_core.messages.ai.AIMessage'>


In [ ]:
# pip install -U langchain-deepseek
from langchain.chat_models import init_chat_model


print(llm.invoke("nihao"))

content='<|im_start|>user\nnihao<|im_end|>\n<|im_start|>assistant\nHello! How can I assist you today?' additional_kwargs={} response_metadata={} id='lc_run--a82ec2a4-9229-4259-83d4-6db0cd45eb6b-0'


# Graph


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)


# Define the function that calls the model
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": response}


workflow.add_node("node1", call_model)

# Define the (single) node in the graph
workflow.add_edge(START, "node1")

# Add memory
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)


In [ ]:
config = {"configurable": {"thread_id": "abc123"}}
query = "Hi! what i name."

input_messages = [HumanMessage(query)]
output = graph.invoke({"messages": input_messages}, config)
output["messages"][1]  # output contains all messages in state

AIMessage(content='<|im_start|>user\nHi! what i name.<|im_end|>\n<|im_start|>assistant\nSure, I can help with that. What would you like to name yourself?', additional_kwargs={}, response_metadata={}, id='lc_run--7eb2d194-5a97-4941-9386-6f1f2856d866-0')

In [ ]:
output["messages"][2]

HumanMessage(content='Hi! what i name.', additional_kwargs={}, response_metadata={}, id='ed599c69-d2ea-4a7c-98e9-13739d1b97c5')

In [ ]:
prompt_template.invoke({"topic": "cat"}).to_messages()[0]

SystemMessage(content='你是一个幽默助手。请按照以下JSON格式返回笑话:\n        {\n            "setup": "笑话的铺垫",\n            "punchline": "笑话的包袱"\n        }\n        只返回JSON格式,不要其他内容。', additional_kwargs={}, response_metadata={})

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from typing import List
from pprint import pprint
from langgraph.types import Send

# Create plan agent
plan_prompt = """You are a research planner. Break down the research task into specific steps.
Format: List 1-3 key questions to answer."""

class QueryAndReason(BaseModel):
    query: str = Field(..., description="检索的 Query，需要符合搜索习惯")
    reason: str = Field(..., description="为什么要检索这个 query")

# Define the structured output format for plan agent
class ResearchPlan(BaseModel):
    """Research plan with key questions"""

    questions: List[QueryAndReason] = Field(
        description="List of 1-3 key research questions to investigate.", max_items=3
    )


plan_agent = ChatDeepSeek(
    model=deepseek_chat_model, api_key=API_KEY
).with_structured_output(ResearchPlan)


pprint(plan_agent.invoke([HumanMessage("write an description about andrej karthy")]))

/tmp/ipykernel_234217/803045248.py:19: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  questions: List[QueryAndReason] = Field(


ResearchPlan(questions=[QueryAndReason(query='Andrej Karpathy biography career achievements', reason="To get comprehensive information about Andrej Karpathy's background, education, and professional accomplishments"), QueryAndReason(query='Andrej Karpathy OpenAI Tesla work contributions', reason='To understand his major roles and contributions at OpenAI and Tesla, including his work on AI and autonomous driving'), QueryAndReason(query='Andrej Karpathy recent projects current work 2024', reason='To find out about his latest activities, projects, and current professional focus')])


In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langgraph.graph import START, END

# Define the structured output format for plan agent
class ResearchPlan(BaseModel):
    """Research plan with key questions"""
    questions: List[str] = Field(
        description="List of 1-3 key research questions to investigate",
        max_items=3
    )

# planner agent 输出 json / markdown （ - [] 第一条要做的事情 -[ ] 第二条要做的事情）
# Create plan agent with structured output
plan_prompt = """You are a research planner. Break down the research task into specific steps.
Your response must be a JSON object with a 'questions' field containing 1-3 key research questions.

Example format:
{
    "questions": [
        "What is X?",
        "How does Y work?",
        "What are the implications of Z?"
    ]
}

Only return the JSON object, no other text."""

plan_agent = create_react_agent(
    model=ChatDeepSeek(model=deepseek_chat_model, api_key=API_KEY),
    tools=[],
    prompt=plan_prompt
)
search_agent = create_react_agent(
    model=ChatDeepSeek(model=deepseek_chat_model, api_key=API_KEY),
    tools=[],
    prompt="you are a helpful assistant. you can search result in search_tool. 当你获得检索结果之后，输出“检索完毕”，必要额外输出任何东西。",
)

report_agent = create_react_agent(
    model=ChatDeepSeek(model=deepseek_chat_model, api_key=API_KEY),
    tools=[],
    prompt="你是一个擅长写作的人。 你可以根据用户的问题，以及检索的结果，生成一个最终的报告。"
)

# plan -> search -> report 

# Create state graph with custom state
class ResearchState(MessagesState):
    """Custom state that includes research plan"""
    plan: ResearchPlan | None = None

# Update workflow with new state
workflow = StateGraph(ResearchState)


# Add nodes and edges with state transformations
workflow.add_node("plan_agent", plan_agent)
workflow.add_node("search_agent", search_agent)
workflow.add_node("report_agent", report_agent)

workflow.add_edge(START, "plan_agent")
workflow.add_edge("plan_agent", "search_agent")
workflow.add_edge("search_agent", "report_agent")
workflow.add_edge("report_agent", END)

# Compile workflow
app = workflow.compile()

/tmp/ipykernel_234217/1762208306.py:8: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  questions: List[str] = Field(
/tmp/ipykernel_234217/1762208306.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  plan_agent = create_react_agent(
/tmp/ipykernel_234217/1762208306.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  search_agent = create_react_agent(
/tmp/ipykernel_234217/1762208306.py:40: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents

In [ ]:
output = app.invoke(
    {"messages": [
        HumanMessage("write an description about andrej karthy")
    ]}
)

In [ ]:
output

{'messages': [HumanMessage(content='write an description about andrej karthy', additional_kwargs={}, response_metadata={}, id='f77055f1-2668-4cc2-a026-ce63e29a32ab'),
  AIMessage(content='{\n    "questions": [\n        "Who is Andrej Karpathy and what are his key professional roles and achievements?",\n        "What are his major contributions to the field of artificial intelligence and deep learning?",\n        "What is his current focus and recent work in AI research or industry?"\n    ]\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 95, 'total_tokens': 158, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 95}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'd6a0adbf-e4c2-4eeb-9e52-ee4828047ed4', 'finish_reason': 'stop', 'l

In [ ]:
for item in output["messages"]:
    item.pretty_print()

================================ Human Message =================================

write an description about andrej karthy
================================== Ai Message ==================================

{
    "questions": [
        "Who is Andrej Karpathy and what are his key professional roles and achievements?",
        "What are his major contributions to the field of artificial intelligence and deep learning?",
        "What is his current focus and recent work in AI research or industry?"
    ]
}
================================== Ai Message ==================================

I will search for information about Andrej Karpathy to provide you with a comprehensive description.  

{
    "questions": [
        "Andrej Karpathy biography and background",
        "Andrej Karpathy contributions to AI and deep learning",
        "Andrej Karpathy current work and recent projects"
    ]
}
================================== Ai Message ==================================

Of course. Here is